# ONTOLOGY ENRICHMENT USING LLMS

In [ ]:
"""
COMPLETE ONTOLOGY ENRICHMENT USING LLMS
References:
- Dong et al. (2024) - Language Model Framework for Concept Placement
- Benis et al. (2022) - MIMO Multilingual Ontology
- He et al. (2023) - DeepOnto for Ontology Engineering
"""

import json
import time
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
import openai
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ============================================
# Data Classes for Enrichment
# ============================================

@dataclass
class ExtractedConcept:
    """New concept extracted from text"""
    name: str
    concept_type: str  # Disease, Symptom, Medication, Procedure, LabTest
    definition: str
    suggested_parents: List[str]
    synonyms: List[str]
    confidence: float
    source_text: str

@dataclass
class GeneratedRelation:
    """Relation between concepts"""
    source: str
    target: str
    relation_type: str  # subclass_of, symptom_of, treated_by, caused_by
    confidence: float
    reasoning: str

@dataclass
class PlacementSuggestion:
    """Suggested placement for new concept"""
    concept: str
    parent_candidates: List[Tuple[str, float]]
    child_candidates: List[Tuple[str, float]]
    is_leaf: bool
    alternative_placements: List[Dict]

# ============================================
# LLM Configuration
# ============================================

class LLMConfig:
    """Configuration for different LLM providers"""

    MODELS = {
        "gpt4": {
            "name": "gpt-4",
            "max_tokens": 8192,
            "cost_per_1k": 0.03,
            "best_for": ["complex_reasoning", "medical_expertise"]
        },


        "llama": {
            "name": "meta-llama/Llama-2-7b-chat-hf",
            "max_tokens": 4096,
            "cost_per_1k": 0.0,
            "best_for": ["local_deployment", "privacy"]
        },
        "gemini": {
            "name": "gemini-pro",
            "max_tokens": 32768,
            "cost_per_1k": 0.001,
            "best_for": ["integration", "speed"]
        }
    }

# ============================================
# 1. CONCEPT EXTRACTION MODULE
# ============================================

class ConceptExtractor:
    """
    Extract new medical concepts from text using LLMs
    Reference: Dong et al. (2023) - Ontology enrichment from texts
    """

    def __init__(self, model_name: str = "gpt-4", api_key: Optional[str] = None):
        self.model_name = model_name
        if api_key:
            openai.api_key = api_key

    def extract_from_text(self, text: str, existing_concepts: List[str],
                          context_window: int = 500) -> List[ExtractedConcept]:
        """
        Extract new concepts from medical text
        """
        prompt = self._build_extraction_prompt(text, existing_concepts, context_window)

        response = self._call_llm(prompt)
        concepts = self._parse_extraction_response(response)

        return concepts

    def _build_extraction_prompt(self, text: str, existing_concepts: List[str],
                                  window: int) -> str:
        """Build extraction prompt for LLM"""

        # Truncate text if needed
        if len(text) > window:
            text = text[:window] + "..."

        return f"""
        You are a medical ontology expert. Extract NEW medical concepts from the following text.

        EXISTING CONCEPTS (DO NOT EXTRACT THESE):
        {', '.join(existing_concepts[:50])}

        TEXT TO ANALYZE:
        {text}

        For each NEW concept, provide:
        1. Concept name (standard medical terminology)
        2. Concept type (choose from: Disease, Symptom, Medication, Procedure, LabTest, Anatomy, ClinicalFinding)
        3. Brief definition (1-2 sentences)
        4. Suggested parent concept (most specific broader concept from existing list)
        5. Synonyms or alternative names
        6. Clinical priority (High/Medium/Low)
        7. Confidence score (0-1)

        Output as JSON array:
        [
            {{
                "name": "...",
                "type": "...",
                "definition": "...",
                "suggested_parent": "...",
                "synonyms": ["..."],
                "priority": "...",
                "confidence": 0.95
            }}
        ]

        Only extract concepts that are truly NEW and clinically relevant.
        """

    def _call_llm(self, prompt: str) -> str:
        """Call LLM API"""
        try:
            response = openai.ChatCompletion.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=2000
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"LLM API Error: {e}")
            return "[]"

    def _parse_extraction_response(self, response: str) -> List[ExtractedConcept]:
        """Parse LLM response into ExtractedConcept objects"""
        try:
            data = json.loads(response)
            concepts = []
            for item in data:
                concept = ExtractedConcept(
                    name=item.get("name", ""),
                    concept_type=item.get("type", "ClinicalFinding"),
                    definition=item.get("definition", ""),
                    suggested_parents=[item.get("suggested_parent", "")],
                    synonyms=item.get("synonyms", []),
                    confidence=item.get("confidence", 0.5),
                    source_text=""
                )
                concepts.append(concept)
            return concepts
        except json.JSONDecodeError:
            return []

    def batch_extract(self, texts: List[str], existing_concepts: List[str]) -> List[ExtractedConcept]:
        """Extract concepts from multiple texts"""
        all_concepts = []
        for text in texts:
            concepts = self.extract_from_text(text, existing_concepts)
            all_concepts.extend(concepts)
            # Update existing concepts to avoid duplicates
            existing_concepts.extend([c.name for c in concepts])
            time.sleep(1)  # Rate limiting
        return all_concepts

# ============================================
# 2. RELATION GENERATION MODULE
# ============================================

class RelationGenerator:
    """
    Generate relationships between concepts using LLMs
    Reference: He et al. (2023) - Language Model Analysis for Subsumption
    """

    def __init__(self, model_name: str = "gpt-4"):
        self.model_name = model_name

    RELATION_TYPES = [
        "subclass_of",      # A is a type of B
        "symptom_of",       # A is symptom of B
        "caused_by",        # A is caused by B
        "treated_by",       # A is treated by B
        "diagnosed_with",   # A diagnoses B
        "associated_with",  # A is associated with B
        "contraindicated",  # A contraindicates B
        "interacts_with"    # A interacts with B
    ]

    def generate_relations(self, concept_a: str, concept_b: str,
                          context: Optional[str] = None) -> List[GeneratedRelation]:
        """
        Generate possible relations between two concepts
        """
        prompt = self._build_relation_prompt(concept_a, concept_b, context)
        response = self._call_llm(prompt)
        relations = self._parse_relation_response(response, concept_a, concept_b)
        return relations

    def _build_relation_prompt(self, a: str, b: str, context: Optional[str]) -> str:
        """Build prompt for relation generation"""
        context_section = f"\nCONTEXT: {context}" if context else ""

        return f"""
        Determine the relationships between these two medical concepts.

        CONCEPT A: {a}
        CONCEPT B: {b}
        {context_section}

        POSSIBLE RELATION TYPES:
        {', '.join(self.RELATION_TYPES)}

        For each possible relation, output:
        1. Relation type
        2. Direction (A→B, B→A, or bidirectional)
        3. Confidence score
        4. Reasoning (medical justification)

        Output as JSON:
        {{
            "relations": [
                {{
                    "type": "subclass_of",
                    "direction": "A→B",
                    "confidence": 0.9,
                    "reasoning": "Medical justification here"
                }}
            ]
        }}
        """

    def _call_llm(self, prompt: str) -> str:
        """Call LLM API for relation generation"""
        response = openai.ChatCompletion.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=1000
        )
        return response.choices[0].message.content

    def _parse_relation_response(self, response: str, a: str, b: str) -> List[GeneratedRelation]:
        """Parse LLM response into GeneratedRelation objects"""
        try:
            data = json.loads(response)
            relations = []
            for rel in data.get("relations", []):
                rel_type = rel.get("type")
                direction = rel.get("direction", "A→B")

                source = a if direction.startswith("A") else b
                target = b if direction.startswith("A") else a

                relation = GeneratedRelation(
                    source=source,
                    target=target,
                    relation_type=rel_type,
                    confidence=rel.get("confidence", 0.5),
                    reasoning=rel.get("reasoning", "")
                )
                relations.append(relation)
            return relations
        except json.JSONDecodeError:
            return []

    def generate_concept_subgraph(self, seed_concept: str,
                                  max_relations: int = 10) -> List[GeneratedRelation]:
        """
        Generate a subgraph of relations around a seed concept
        """
        prompt = f"""
        Generate a network of medical relationships for: {seed_concept}

        Include:
        - Parents (broader concepts)
        - Children (narrower concepts)
        - Symptoms
        - Treatments
        - Diagnostic tests
        - Related conditions

        Output as JSON array of relations:
        [
            {{
                "source": "{seed_concept}",
                "target": "concept_name",
                "type": "relation_type",
                "confidence": 0.9,
                "reasoning": "..."
            }}
        ]
        """

        response = self._call_llm(prompt)
        try:
            data = json.loads(response)
            relations = []
            for item in data[:max_relations]:
                rel = GeneratedRelation(
                    source=item.get("source", seed_concept),
                    target=item.get("target", ""),
                    relation_type=item.get("type", "associated_with"),
                    confidence=item.get("confidence", 0.5),
                    reasoning=item.get("reasoning", "")
                )
                relations.append(rel)
            return relations
        except json.JSONDecodeError:
            return []

# ============================================
# 3. PLACEMENT SUGGESTION MODULE
# ============================================

class PlacementSuggester:
    """
    Suggest optimal placement for new concepts in ontology
    Reference: Dong et al. (2024) - Edge-Bi-encoder framework
    Reference: Liu et al. (2020) - Concept Placement using BERT
    """

    def __init__(self, model_name: str = "gpt-4"):
        self.model_name = model_name

    def suggest_placement(self, new_concept: ExtractedConcept,
                          available_parents: List[str],
                          available_children: List[str],
                          context: Optional[str] = None) -> PlacementSuggestion:
        """
        Suggest where to place new concept in ontology hierarchy
        """
        prompt = self._build_placement_prompt(
            new_concept, available_parents, available_children, context
        )
        response = self._call_llm(prompt)
        suggestion = self._parse_placement_response(response, new_concept.name)
        return suggestion

    def _build_placement_prompt(self, concept: ExtractedConcept,
                                 parents: List[str],
                                 children: List[str],
                                 context: Optional[str]) -> str:
        """Build prompt for placement suggestion"""
        return f"""
        Place this new medical concept into the existing ontology hierarchy.

        NEW CONCEPT: {concept.name}
        CONCEPT TYPE: {concept.concept_type}
        DEFINITION: {concept.definition}
        {f'CONTEXT: {context}' if context else ''}

        AVAILABLE PARENTS (broader concepts that could be parents):
        {', '.join(parents[:30])}

        AVAILABLE CHILDREN (narrower concepts that could be children):
        {', '.join(children[:30])}

        Task: Suggest the BEST placement(s) for this concept.

        Consider:
        1. Most appropriate parent (most specific broader concept)
        2. Most appropriate child (most general narrower concept)
        3. Whether it should be a leaf node (no children)
        4. Whether it needs multiple parents (multiple inheritance)
        5. Clinical relevance of placement

        Output as JSON:
        {{
            "parent_candidates": [
                {{"parent": "parent_name", "score": 0.95, "reasoning": "..."}}
            ],
            "child_candidates": [
                {{"child": "child_name", "score": 0.90, "reasoning": "..."}}
            ],
            "is_leaf": false,
            "alternative_placements": [
                {{"parent": "...", "child": "...", "score": 0.8}}
            ],
            "overall_confidence": 0.92
        }}
        """

    def _call_llm(self, prompt: str) -> str:
        """Call LLM for placement suggestion"""
        response = openai.ChatCompletion.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=1500
        )
        return response.choices[0].message.content

    def _parse_placement_response(self, response: str, concept_name: str) -> PlacementSuggestion:
        """Parse LLM response into PlacementSuggestion"""
        try:
            data = json.loads(response)

            parent_candidates = [
                (p["parent"], p["score"])
                for p in data.get("parent_candidates", [])
            ]
            child_candidates = [
                (c["child"], c["score"])
                for c in data.get("child_candidates", [])
            ]

            return PlacementSuggestion(
                concept=concept_name,
                parent_candidates=parent_candidates,
                child_candidates=child_candidates,
                is_leaf=data.get("is_leaf", False),
                alternative_placements=data.get("alternative_placements", [])
            )
        except json.JSONDecodeError:
            return PlacementSuggestion(
                concept=concept_name,
                parent_candidates=[],
                child_candidates=[],
                is_leaf=True,
                alternative_placements=[]
            )

    def batch_suggest(self, concepts: List[ExtractedConcept],
                     ontology_structure: Dict) -> List[PlacementSuggestion]:
        """
        Suggest placements for multiple concepts
        """
        suggestions = []
        for concept in concepts:
            suggestion = self.suggest_placement(
                concept,
                ontology_structure.get("parents", []),
                ontology_structure.get("children", [])
            )
            suggestions.append(suggestion)
            time.sleep(0.5)
        return suggestions

# ============================================
# 4. MULTILINGUAL EXPANSION MODULE
# ============================================

class MultilingualExpander:
    """
    Expand ontology with multilingual support using LLMs
    Reference: Benis et al. (2022) - MIMO multilingual ontology
    """

    def __init__(self, model_name: str = "gpt-4"):
        self.model_name = model_name

    SUPPORTED_LANGUAGES = {

        "en": "English"

    }

    def translate_concept(self, concept_name: str, target_language: str) -> Dict:
        """
        Translate a medical concept to target language
        """
        prompt = f"""
        Translate this medical concept to {self.SUPPORTED_LANGUAGES.get(target_language, target_language)}.

        CONCEPT: {concept_name}

        Requirements:
        1. Use accurate medical terminology
        2. Provide multiple translation options if applicable
        3. Include confidence score
        4. Note any cultural or regional variations

        Output as JSON:
        {{
            "original": "{concept_name}",
            "translation": "translated_text",
            "alternative_translations": ["alt1", "alt2"],
            "confidence": 0.95,
            "regional_variants": {{
                "region1": "variant1",
                "region2": "variant2"
            }}
        }}
        """

        response = openai.ChatCompletion.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=500
        )

        try:
            return json.loads(response.choices[0].message.content)
        except:
            return {"original": concept_name, "translation": concept_name, "confidence": 0.5}

    def batch_translate(self, concepts: List[str], target_languages: List[str]) -> Dict:
        """
        Translate multiple concepts to multiple languages
        """
        translations = {}
        for concept in concepts:
            translations[concept] = {}
            for lang in target_languages:
                trans = self.translate_concept(concept, lang)
                translations[concept][lang] = trans
                time.sleep(0.3)
        return translations

    def add_cultural_context(self, concept_name: str, region: str) -> Dict:
        """
        Add region-specific cultural context to concepts
        """
        prompt = f"""
        Add cultural and regional context for the medical concept '{concept_name}' in {region}.

        Provide:
        1. Local terminology or common names
        2. Regional prevalence or epidemiological data
        3. Local treatment preferences
        4. Cultural considerations for patient communication
        5. Local healthcare system context

        Output as JSON:
        {{
            "concept": "{concept_name}",
            "region": "{region}",
            "local_terms": ["term1", "term2"],
            "prevalence": "High/Medium/Low/Unknown",
            "cultural_notes": "Important cultural considerations",
            "local_treatment": "Common local treatment approaches"
        }}
        """

        response = openai.ChatCompletion.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4,
            max_tokens=800
        )

        try:
            return json.loads(response.choices[0].message.content)
        except:
            return {"concept": concept_name, "region": region}

# ============================================
# 5. MAIN ENRICHMENT ORCHESTRATOR
# ============================================

class OntologyEnrichmentOrchestrator:
    """
    Complete orchestration of ontology enrichment using LLMs
    """

    def __init__(self, output_dir: str = "./enriched_ontology"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Initialize components
        self.concept_extractor = ConceptExtractor(model_name="gpt-4")
        self.relation_generator = RelationGenerator(model_name="gpt-4")
        self.placement_suggester = PlacementSuggester(model_name="gpt-4")
        self.multilingual_expander = MultilingualExpander(model_name="gpt-4")

        # Statistics tracking
        self.stats = {
            "concepts_extracted": 0,
            "relations_generated": 0,
            "placements_suggested": 0,
            "translations_added": 0,
            "total_cost_usd": 0.0
        }

    def run_complete_enrichment(self, source_texts: List[str],
                                existing_ontology: Dict,
                                target_languages: List[str] = ["ar", "en", "fr", "de", "es"]) -> Dict:
        """
        Run complete ontology enrichment pipeline
        """
        print("\n" + "="*70)
        print(" ONTOLOGY ENRICHMENT PIPELINE")
        print("="*70)

        results = {
            "new_concepts": [],
            "new_relations": [],
            "placements": [],
            "translations": {},
            "cultural_contexts": []
        }

        # ========== STEP 1: Extract New Concepts ==========
        print("\n STEP 1: Extracting New Concepts from Texts")
        print("-"*40)

        existing_concepts = existing_ontology.get("concepts", [])
        new_concepts = self.concept_extractor.batch_extract(source_texts, existing_concepts)
        results["new_concepts"] = new_concepts
        self.stats["concepts_extracted"] = len(new_concepts)
        print(f"   ✓ Extracted {len(new_concepts)} new concepts")

        for i, concept in enumerate(new_concepts[:5], 1):
            print(f"      {i}. {concept.name} ({concept.concept_type}) - {concept.definition[:60]}...")

        # ========== STEP 2: Generate Relations ==========
        print("\n🔗 STEP 2: Generating Relations")
        print("-"*40)

        all_relations = []
        for concept in new_concepts[:10]:  # Limit for demonstration
            for existing in existing_concepts[:5]:
                relations = self.relation_generator.generate_relations(concept.name, existing)
                all_relations.extend(relations)

        results["new_relations"] = all_relations
        self.stats["relations_generated"] = len(all_relations)
        print(f"   ✓ Generated {len(all_relations)} relations")

        # ========== STEP 3: Suggest Placements ==========
        print("\n STEP 3: Suggesting Ontology Placements")
        print("-"*40)

        ontology_structure = {
            "parents": existing_concepts[:50],
            "children": existing_concepts[50:100] if len(existing_concepts) > 50 else []
        }

        placements = self.placement_suggester.batch_suggest(new_concepts[:10], ontology_structure)
        results["placements"] = placements
        self.stats["placements_suggested"] = len(placements)
        print(f"   ✓ Suggested placements for {len(placements)} concepts")

        for placement in placements[:3]:
            if placement.parent_candidates:
                print(f"      - {placement.concept} → parent: {placement.parent_candidates[0][0]} (score: {placement.parent_candidates[0][1]})")

        # ========== STEP 4: Multilingual Expansion ==========
        print("\n STEP 4: Multilingual Expansion")
        print("-"*40)

        # Translate new concepts
        concept_names = [c.name for c in new_concepts[:10]]
        translations = self.multilingual_expander.batch_translate(concept_names, target_languages)
        results["translations"] = translations
        self.stats["translations_added"] = sum(len(t) for t in translations.values())
        print(f"   ✓ Added {self.stats['translations_added']} translations")

        # Add cultural context for Arabic region
        for concept in new_concepts[:3]:
            cultural = self.multilingual_expander.add_cultural_context(concept.name, "Middle East")
            results["cultural_contexts"].append(cultural)
        print(f"   ✓ Added cultural context for {len(results['cultural_contexts'])} concepts")

        # ========== STEP 5: Generate Report ==========
        print("\n STEP 5: Generating Enrichment Report")
        print("-"*40)

        report_path = self._generate_report(results)
        print(f"   ✓ Report saved: {report_path}")

        # ========== SUMMARY ==========
        print("\n" + "="*70)
        print(" ONTOLOGY ENRICHMENT COMPLETED")
        print("="*70)
        print(f"\n📈 STATISTICS:")
        print(f"   • New Concepts: {self.stats['concepts_extracted']}")
        print(f"   • New Relations: {self.stats['relations_generated']}")
        print(f"   • Placement Suggestions: {self.stats['placements_suggested']}")
        print(f"   • Translations Added: {self.stats['translations_added']}")
        print(f"\n Output Directory: {self.output_dir.absolute()}")
        print("="*70)

        return results

    def _generate_report(self, results: Dict) -> Path:
        """Generate comprehensive enrichment report"""

        report_path = self.output_dir / "ONTOLOGY_ENRICHMENT_REPORT.md"

        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("# Ontology Enrichment Report\n\n")
            f.write(f"**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

            # Summary
            f.write("## Executive Summary\n\n")
            f.write(f"- **New Concepts Extracted:** {len(results['new_concepts'])}\n")
            f.write(f"- **New Relations Generated:** {len(results['new_relations'])}\n")
            f.write(f"- **Placement Suggestions:** {len(results['placements'])}\n")
            f.write(f"- **Translations Added:** {self.stats['translations_added']}\n\n")

            # New Concepts Table
            f.write("## New Concepts\n\n")
            f.write("| # | Concept | Type | Definition | Confidence |\n")
            f.write("|---|---------|------|------------|------------|\n")
            for i, concept in enumerate(results['new_concepts'][:20], 1):
                f.write(f"| {i} | {concept.name} | {concept.concept_type} | {concept.definition[:50]}... | {concept.confidence} |\n")

            # New Relations
            f.write("\n## New Relations\n\n")
            f.write("| Source | Target | Relation Type | Confidence |\n")
            f.write("|--------|--------|---------------|------------|\n")
            for rel in results['new_relations'][:20]:
                f.write(f"| {rel.source} | {rel.target} | {rel.relation_type} | {rel.confidence} |\n")

            # Placements
            f.write("\n## Placement Suggestions\n\n")
            for placement in results['placements'][:10]:
                f.write(f"\n### {placement.concept}\n\n")
                if placement.parent_candidates:
                    f.write(f"**Suggested Parent:** {placement.parent_candidates[0][0]} (score: {placement.parent_candidates[0][1]})\n\n")
                if placement.child_candidates:
                    f.write(f"**Suggested Child:** {placement.child_candidates[0][0]} (score: {placement.child_candidates[0][1]})\n\n")
                f.write(f"**Is Leaf Node:** {placement.is_leaf}\n\n")

            # Translations
            f.write("\n## Multilingual Translations\n\n")
            for concept, trans in list(results['translations'].items())[:5]:
                f.write(f"\n### {concept}\n\n")
                for lang, t in trans.items():
                    f.write(f"- **{lang.upper()}:** {t.get('translation', 'N/A')} (confidence: {t.get('confidence', 0)})\n")

        return report_path

# ============================================
# 6. VALIDATION MODULE
# ============================================

class EnrichmentValidator:
    """
    Validate enriched ontology content
    Reference: Dong et al. (2024) - Validation framework
    """

    @staticmethod
    def validate_consistency(new_concepts: List[ExtractedConcept],
                            existing_concepts: List[str]) -> List[Dict]:
        """
        Check for duplicates and inconsistencies
        """
        issues = []

        # Check for duplicates
        concept_names = [c.name.lower() for c in new_concepts]
        duplicates = [name for name in concept_names if concept_names.count(name) > 1]
        if duplicates:
            issues.append({
                "type": "duplicate_concepts",
                "details": f"Duplicate concepts found: {set(duplicates)}"
            })

        # Check against existing concepts
        existing_lower = [e.lower() for e in existing_concepts]
        for concept in new_concepts:
            if concept.name.lower() in existing_lower:
                issues.append({
                    "type": "already_exists",
                    "concept": concept.name,
                    "details": f"Concept already exists in ontology"
                })

        # Check confidence threshold
        low_confidence = [c.name for c in new_concepts if c.confidence < 0.7]
        if low_confidence:
            issues.append({
                "type": "low_confidence",
                "concepts": low_confidence,
                "details": "Concepts with confidence < 0.7 need human review"
            })

        return issues

    @staticmethod
    def calculate_quality_metrics(results: Dict) -> Dict:
        """
        Calculate quality metrics for enrichment
        """
        metrics = {
            "total_concepts": len(results.get("new_concepts", [])),
            "average_confidence": 0,
            "relation_density": 0,
            "coverage_score": 0
        }

        # Average confidence
        concepts = results.get("new_concepts", [])
        if concepts:
            metrics["average_confidence"] = sum(c.confidence for c in concepts) / len(concepts)

        # Relation density (relations per concept)
        relations = results.get("new_relations", [])
        if concepts:
            metrics["relation_density"] = len(relations) / len(concepts)

        return metrics

# ============================================
# 7. MAIN EXECUTION
# ============================================

def main():
    """
    Main execution function
    """
    print("\n" + "="*70)
    print(" HEALTH LEARNING ONTOLOGY (HLO) ENRICHMENT")
    print("Using Large Language Models for Ontology Enrichment")
    print("="*70)

    # Sample medical texts for extraction
    sample_texts = [
        """
        Long COVID, also known as post-acute sequelae of SARS-CoV-2 infection (PASC),
        is characterized by persistent symptoms lasting more than 12 weeks after initial infection.
        Common symptoms include severe fatigue, cognitive impairment ('brain fog'),
        dyspnea, chest pain, and palpitations. Management requires multidisciplinary approach
        including symptom-based treatment and rehabilitation.
        """,

        """
        Digital therapeutics (DTx) are evidence-based therapeutic interventions delivered
        through software applications to prevent, manage, or treat medical conditions.
        They have shown efficacy in diabetes management, hypertension control,
        and mental health disorders including depression and anxiety.
        """,

        """
        Artificial intelligence in medical education is transforming how residents learn.
        AI-powered tools can provide personalized learning paths, real-time feedback,
        and simulation-based training. Key applications include diagnostic assistance,
        treatment recommendations, and automated assessment of clinical skills.
        """
    ]

    # Mock existing ontology concepts
    existing_concepts = [
        "COVID-19", "SARS-CoV-2 infection", "Fatigue", "Dyspnea", "Chest pain",
        "Diabetes mellitus", "Hypertension", "Depression", "Anxiety disorder",
        "Medical education", "Clinical skills", "Diagnostic imaging"
    ]

    # Initialize orchestrator
    orchestrator = OntologyEnrichmentOrchestrator(output_dir="./hlo_enrichment_output")

    # Run enrichment
    results = orchestrator.run_complete_enrichment(
        source_texts=sample_texts,
        existing_ontology={"concepts": existing_concepts},
        target_languages=["en"]
    )

    # Validate results
    print("\n VALIDATING ENRICHMENT RESULTS")
    print("-"*40)

    issues = EnrichmentValidator.validate_consistency(
        results["new_concepts"],
        existing_concepts
    )

    if issues:
        print(" Validation Issues Found:")
        for issue in issues:
            print(f"   • {issue['type']}: {issue.get('details', '')}")
    else:
        print(" No validation issues detected")

    metrics = EnrichmentValidator.calculate_quality_metrics(results)
    print(f"\n Quality Metrics:")
    print(f"   • Average Confidence: {metrics['average_confidence']:.2f}")
    print(f"   • Relation Density: {metrics['relation_density']:.2f}")

    print("\n" + "="*70)
    print(" ENRICHMENT COMPLETE!")
    print(f" Output: ./hlo_enrichment_output/")
    print("="*70)

if __name__ == "__main__":
    main()


 HEALTH LEARNING ONTOLOGY (HLO) ENRICHMENT
Using Large Language Models for Ontology Enrichment

 ONTOLOGY ENRICHMENT PIPELINE

 STEP 1: Extracting New Concepts from Texts
----------------------------------------
LLM API Error: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

LLM API Error: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can p

In [ ]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 55.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.50-cp312-cp312-linux_x86_64.whl size=24567085 sha256=7f71ae67b6d7bd06176e9977f495332f7124ea719a7270318f9d030ef7878311
  Stored in directory: /root/.cache/pip/wheels/fb/9d/9d/24582de3856fe0ff897408eb6ed1baa6cd5b7b112a2b9c711c
Successfully built owlready2


# Concept Generation Module

In [ ]:
"""
PROFESSIONAL ONTOLOGY ENRICHMENT SYSTEM
Complete pipeline for concept generation and learning resource linking
References:
- Dong et al. (2024) - LLM Framework for Concept Placement
- Benis et al. (2022) - MIMO Multilingual Ontology
- He et al. (2023) - DeepOnto for Relation Prediction
"""

import json
import time
from typing import List, Dict, Set, Tuple, Optional
from dataclasses import dataclass, field
from pathlib import Path
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from owlready2 import *

# ============================================
# DATA CLASSES
# ============================================

@dataclass
class GeneratedConcept:
    """New concept generated from text"""
    name: str
    concept_type: str  # Disease, Symptom, Medication, Procedure, LabTest
    definition: str
    synonyms: List[str]
    confidence: float
    source_text: str
    suggested_parents: List[str] = field(default_factory=list)
    suggested_children: List[str] = field(default_factory=list)

@dataclass
class LearningResourceLink:
    """Link between concept and learning resource"""
    concept_id: str
    concept_name: str
    resource_id: str
    resource_title: str
    resource_type: str  # Video, Article, Guideline
    difficulty_level: str  # Beginner, Intermediate, Advanced
    recommended_levels: List[str]  # PGY1, PGY2, PGY3, PGY4, Fellow
    match_score: float
    reasoning: str

@dataclass
class EnrichmentResult:
    """Complete enrichment result"""
    new_concepts: List[GeneratedConcept]
    concept_relations: List[Tuple[str, str, str]]  # source, target, relation_type
    resource_links: List[LearningResourceLink]
    cross_mappings: Dict[str, str]
    validation_metrics: Dict

# ============================================
# CONCEPT GENERATOR
# ============================================

class ProfessionalConceptGenerator:
    """
    Generate new medical concepts from texts using LLMs
    Reference: Dong et al. (2023) - Ontology enrichment from texts
    """

    def __init__(self, llm_model: str = "gpt-4", api_key: Optional[str] = None):
        self.llm_model = llm_model
        if api_key:
            import openai
            openai.api_key = api_key

        # Initialize biomedical embeddings
        self.tokenizer = AutoTokenizer.from_pretrained(
            "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
        )
        self.encoder = AutoModel.from_pretrained(
            "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
        )

    def extract_concepts_from_text(self, text: str,
                                   existing_concepts: Set[str],
                                   context: str = "") -> List[GeneratedConcept]:
        """
        Extract new concepts from medical text using LLM
        """
        print(f"\n🔍 Extracting concepts from text...")

        # Build prompt for LLM
        prompt = self._build_extraction_prompt(text, existing_concepts, context)

        # Call LLM
        response = self._call_llm(prompt)

        # Parse and validate concepts
        concepts = self._parse_concepts(response, text)

        # Validate against existing ontology
        validated = self._validate_concepts(concepts, existing_concepts)

        return validated

    def _build_extraction_prompt(self, text: str, existing: Set[str], context: str) -> str:
        """Build professional extraction prompt"""
        existing_sample = list(existing)[:100] if existing else []

        return f"""
        You are a medical ontology expert. Extract NEW medical concepts from the following text.

        IMPORTANT: Only extract concepts that are NOT in the existing ontology.

        EXISTING CONCEPTS (DO NOT EXTRACT THESE):
        {', '.join(existing_sample[:50])}

        TEXT TO ANALYZE:
        {text[:3000]}

        {f'CONTEXT: {context}' if context else ''}

        For each NEW concept, provide:
        1. concept_name (standard medical terminology)
        2. concept_type (choose from: Disease, Symptom, Medication, Procedure, LabTest, Anatomy, ClinicalFinding)
        3. definition (1-2 sentences)
        4. synonyms (alternative names)
        5. suggested_parents (most specific broader concepts from existing list)
        6. confidence (0-1)

        Output as JSON array:
        [
            {{
                "concept_name": "...",
                "concept_type": "...",
                "definition": "...",
                "synonyms": ["..."],
                "suggested_parents": ["..."],
                "confidence": 0.95
            }}
        ]

        Only extract clinically relevant concepts with confidence > 0.7.
        """

    def _call_llm(self, prompt: str) -> str:
        """Call LLM API with retry logic"""
        try:
            import openai
            response = openai.ChatCompletion.create(
                model=self.llm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=2000
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"   ⚠️ LLM API error: {e}")
            return "[]"

    def _parse_concepts(self, response: str, source_text: str) -> List[GeneratedConcept]:
        """Parse LLM response into GeneratedConcept objects"""
        concepts = []
        try:
            data = json.loads(response)
            for item in data:
                concept = GeneratedConcept(
                    name=item.get("concept_name", ""),
                    concept_type=item.get("concept_type", "ClinicalFinding"),
                    definition=item.get("definition", ""),
                    synonyms=item.get("synonyms", []),
                    confidence=item.get("confidence", 0.5),
                    source_text=source_text[:200],
                    suggested_parents=item.get("suggested_parents", [])
                )
                if concept.name and concept.confidence > 0.7:
                    concepts.append(concept)
        except json.JSONDecodeError:
            print(f"   ⚠️ Failed to parse LLM response")

        return concepts

    def _validate_concepts(self, concepts: List[GeneratedConcept],
                          existing: Set[str]) -> List[GeneratedConcept]:
        """Validate concepts against existing ontology"""
        validated = []
        existing_lower = {e.lower() for e in existing}

        for concept in concepts:
            # Check if already exists
            if concept.name.lower() in existing_lower:
                print(f"   ⚠️ Skipping duplicate: {concept.name}")
                continue

            # Check for synonyms that might exist
            duplicate_synonym = False
            for syn in concept.synonyms:
                if syn.lower() in existing_lower:
                    duplicate_synonym = True
                    print(f"   ⚠️ Synonym already exists: {syn} → {concept.name}")
                    break

            if not duplicate_synonym:
                validated.append(concept)
                print(f"   ✅ Validated: {concept.name} ({concept.concept_type})")

        return validated

    def compute_semantic_similarity(self, concept1: str, concept2: str) -> float:
        """Compute semantic similarity using PubMedBERT"""
        inputs1 = self.tokenizer(concept1, return_tensors="pt", truncation=True, max_length=128)
        inputs2 = self.tokenizer(concept2, return_tensors="pt", truncation=True, max_length=128)

        with torch.no_grad():
            emb1 = self.encoder(**inputs1).last_hidden_state[:, 0, :]
            emb2 = self.encoder(**inputs2).last_hidden_state[:, 0, :]

        similarity = torch.cosine_similarity(emb1, emb2).item()
        return similarity

    def batch_extract(self, texts: List[Dict],
                     existing_concepts: Set[str]) -> List[GeneratedConcept]:
        """
        Batch extract concepts from multiple texts
        """
        all_concepts = []

        for i, text_data in enumerate(texts):
            print(f"\n📄 Processing text {i+1}/{len(texts)}")
            concepts = self.extract_concepts_from_text(
                text_data.get("content", ""),
                existing_concepts,
                text_data.get("context", "")
            )
            all_concepts.extend(concepts)

            # Update existing concepts to avoid duplicates
            for c in concepts:
                existing_concepts.add(c.name)
                existing_concepts.update(c.synonyms)

            time.sleep(0.5)  # Rate limiting

        return all_concepts

# Relation Generation Module

In [ ]:
class RelationGenerator:
    """
    Generate hierarchical and associative relations between concepts
    Reference: He et al. (2023) - Subsumption prediction
    """

    def __init__(self):
        self.relation_types = [
            "subclass_of",      # A is a type of B
            "symptom_of",       # A is symptom of B
            "caused_by",        # A is caused by B
            "treated_by",       # A is treated by B
            "diagnosed_by",     # A diagnosed by B
            "associated_with",  # A associated with B
            "contraindicated",  # A contraindicates B
            "part_of"          # A is part of B
        ]

    def generate_parent_relations(self, concept: GeneratedConcept,
                                  existing_concepts: List[str]) -> List[Tuple[str, float]]:
        """
        Generate parent (broader) relations for a new concept
        """
        print(f"   Generating parents for: {concept.name}")

        parents = []

        # Rule-based matching first
        for existing in existing_concepts:
            # Check if existing concept contains the new concept
            if concept.name.lower() in existing.lower():
                parents.append((existing, 0.9))
            # Check if new concept is a subtype of existing
            elif self._is_subtype(concept.name, existing):
                parents.append((existing, 0.8))

        # Add concept.suggested_parents
        for suggested in concept.suggested_parents:
            if suggested in existing_concepts:
                parents.append((suggested, 0.85))

        # Remove duplicates and sort
        unique_parents = {}
        for parent, score in parents:
            if parent not in unique_parents or score > unique_parents[parent]:
                unique_parents[parent] = score

        return sorted(unique_parents.items(), key=lambda x: x[1], reverse=True)

    def _is_subtype(self, concept: str, parent: str) -> bool:
        """Check if concept is a subtype of parent"""
        # Common subtype patterns
        subtype_indicators = [
            f"{parent} due to",
            f"{parent} caused by",
            f"{parent} associated with",
            f"acute {parent}",
            f"chronic {parent}",
            f"severe {parent}"
        ]

        concept_lower = concept.lower()
        for indicator in subtype_indicators:
            if indicator.lower() in concept_lower:
                return True

        return False

    def generate_child_relations(self, concept: GeneratedConcept,
                                 existing_concepts: List[str]) -> List[Tuple[str, float]]:
        """
        Generate child (narrower) relations for a new concept
        """
        print(f"   Generating children for: {concept.name}")

        children = []

        for existing in existing_concepts:
            # If existing concept contains the new concept name
            if existing.lower() in concept.name.lower():
                children.append((existing, 0.85))
            # Check if existing is a variation
            elif self._is_variation(existing, concept.name):
                children.append((existing, 0.75))

        return sorted(children, key=lambda x: x[1], reverse=True)[:10]

    def _is_variation(self, candidate: str, base: str) -> bool:
        """Check if candidate is a variation of base concept"""
        variations = [
            f"{base} with",
            f"{base} without",
            f"{base} stage",
            f"{base} type",
            f"{base} syndrome"
        ]

        candidate_lower = candidate.lower()
        base_lower = base.lower()

        for var in variations:
            if var.lower() in candidate_lower and base_lower in candidate_lower:
                return True

        return False

    def generate_sibling_relations(self, concept: GeneratedConcept,
                                   parents: List[str],
                                   ontology_graph: Dict) -> List[str]:
        """
        Generate sibling concepts (share same parent)
        """
        siblings = set()

        for parent in parents:
            if parent in ontology_graph:
                siblings.update(ontology_graph[parent].get("children", []))

        # Remove self
        siblings.discard(concept.name)

        return list(siblings)[:15]

    def generate_associative_relations(self, concept1: str, concept2: str) -> List[str]:
        """
        Generate associative relations between two concepts
        """
        relations = []

        # Disease-Symptom relations
        if "disease" in concept1.lower() and "symptom" in concept2.lower():
            relations.append("causes")
        elif "symptom" in concept1.lower() and "disease" in concept2.lower():
            relations.append("symptom_of")

        # Disease-Treatment relations
        if "disease" in concept1.lower() and "medication" in concept2.lower():
            relations.append("treated_by")
        elif "medication" in concept1.lower() and "disease" in concept2.lower():
            relations.append("treats")

        # Disease-Diagnosis relations
        if "disease" in concept1.lower() and "procedure" in concept2.lower():
            relations.append("diagnosed_by")

        return relations

#  Learning Resource Linking Module

In [ ]:
class LearningResourceLinker:
    """
    Link generated concepts to appropriate learning resources
    Reference: ACGME Milestones for level-based recommendations
    """

    def __init__(self, resources_db: Dict):
        self.resources_db = resources_db
        self.resource_templates = self._init_templates()

    def _init_templates(self) -> Dict:
        """Initialize resource templates for new concepts"""
        return {
            "Video": {
                "template": "Understanding {concept_name}",
                "duration": 15,
                "difficulty": "Beginner",
                "levels": ["PGY1", "PGY2"]
            },
            "Article": {
                "template": "Clinical Management of {concept_name}",
                "duration": 30,
                "difficulty": "Intermediate",
                "levels": ["PGY2", "PGY3"]
            },
            "ClinicalGuideline": {
                "template": "Evidence-Based Guidelines for {concept_name}",
                "duration": 45,
                "difficulty": "Advanced",
                "levels": ["PGY3", "PGY4", "Fellow"]
            },
            "CaseStudy": {
                "template": "Case Study: {concept_name}",
                "duration": 20,
                "difficulty": "Intermediate",
                "levels": ["PGY2", "PGY3"]
            }
        }

    def link_concept_to_resources(self, concept: GeneratedConcept,
                                  priority: str = "High") -> List[LearningResourceLink]:
        """
        Link a generated concept to appropriate learning resources
        """
        print(f"\n🔗 Linking resources for: {concept.name}")

        links = []

        # Determine resource types based on priority
        if priority == "High":
            resource_types = ["Video", "Article", "ClinicalGuideline"]
        elif priority == "Medium":
            resource_types = ["Video", "Article"]
        else:
            resource_types = ["Video"]

        # Generate resources for each type
        for res_type in resource_types:
            template = self.resource_templates.get(res_type, self.resource_templates["Video"])

            # Create resource link
            link = LearningResourceLink(
                concept_id=f"CONCEPT_{concept.name.replace(' ', '_')}",
                concept_name=concept.name,
                resource_id=f"RES_{res_type}_{concept.name.replace(' ', '_')[:10]}",
                resource_title=template["template"].format(concept_name=concept.name),
                resource_type=res_type,
                difficulty_level=template["difficulty"],
                recommended_levels=template["levels"],
                match_score=self._calculate_match_score(concept, res_type),
                reasoning=f"Educational resource for learning {concept.name} at {template['difficulty']} level"
            )
            links.append(link)

            print(f"   ✅ Created {res_type}: {link.resource_title}")

        return links

    def _calculate_match_score(self, concept: GeneratedConcept, resource_type: str) -> float:
        """Calculate match score between concept and resource type"""
        base_score = concept.confidence

        # Adjust based on resource type appropriateness
        type_scores = {
            "Video": 1.0,
            "Article": 0.9,
            "ClinicalGuideline": 0.85,
            "CaseStudy": 0.8
        }

        return base_score * type_scores.get(resource_type, 0.8)

    def assign_difficulty_by_level(self, concept: GeneratedConcept,
                                   level: str) -> str:
        """
        Assign difficulty level based on PGY level
        """
        level_difficulty = {
            "PGY1": "Beginner",
            "PGY2": "Beginner",
            "PGY3": "Intermediate",
            "PGY4": "Advanced",
            "Fellow": "Advanced"
        }
        return level_difficulty.get(level, "Intermediate")

    def batch_link(self, concepts: List[GeneratedConcept]) -> List[LearningResourceLink]:
        """
        Batch link multiple concepts to resources
        """
        all_links = []
        for concept in concepts:
            # Determine priority based on concept type
            priority = "High" if concept.concept_type == "Disease" else "Medium"
            links = self.link_concept_to_resources(concept, priority)
            all_links.extend(links)

        return all_links

# Ontology Enrichment

In [ ]:
!rm -r LM-ontology-concept-placement # remove if existed previously
!git clone https://github.com/KRR-Oxford/LM-ontology-concept-placement.git


rm: cannot remove 'LM-ontology-concept-placement': No such file or directory
Cloning into 'LM-ontology-concept-placement'...
remote: Enumerating objects: 279, done.
remote: Counting objects: 100% (279/279), done.
remote: Compressing objects: 100% (237/237), done.
remote: Total 279 (delta 97), reused 191 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (279/279), 567.33 KiB | 5.73 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [ ]:
!pip install zenodo_get # install zenodo
!zenodo_get 10432003 # get dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.5 MB/s eta 0:00:00
INFO: Output directory: /content
INFO: Title: Ontology Enrichment from Texts (OET): A Biomedical Dataset for Concept Discovery and Placement
INFO: Total size: 595.4 MB
INFO: Number of files: 1
SUCCESS: All specified files have been processed.


In [ ]:
!rm -r LM-ontology-concept-placement/data # remove if existed previously
!mkdir LM-ontology-concept-placement/data
!unzip OET-data-ver4.zip -d LM-ontology-concept-placement/data

rm: cannot remove 'LM-ontology-concept-placement/data': No such file or directory
Archive:  OET-data-ver4.zip
  inflating: LM-ontology-concept-placement/data/readme.md  
  inflating: LM-ontology-concept-placement/data/__MACOSX/._readme.md  
   creating: LM-ontology-concept-placement/data/MM-S14-Disease/
  inflating: LM-ontology-concept-placement/data/__MACOSX/._MM-S14-Disease  
  inflating: LM-ontology-concept-placement/data/MM-S14-Disease/.DS_Store  
  inflating: LM-ontology-concept-placement/data/__MACOSX/MM-S14-Disease/._.DS_Store  
   creating: LM-ontology-concept-placement/data/MM-S14-Disease/mention-edge-pair-level/
   creating: LM-ontology-concept-placement/data/MM-S14-Disease/mention-level-(out-of-KB-discovery)/
   creating: LM-ontology-concept-placement/data/MM-S14-Disease/mention-level-(concept-placement)/
   creating: LM-ontology-concept-placement/data/MM-S14-Disease/ontology/
  inflating: LM-ontology-concept-placement/data/MM-S14-Disease/mention-edge-pair-level/train.jsonl 

In [ ]:
# rename data files
!mv "LM-ontology-concept-placement/data/MM-S14-Disease/mention-level-(out-of-KB-discovery)/syn_attr" "LM-ontology-concept-placement/data/MM-S14-Disease/st21pv_syn_attr-all-complexEdge-filt"
!mv "LM-ontology-concept-placement/data/MM-S14-Disease/mention-level-(out-of-KB-discovery)/syn_full" "LM-ontology-concept-placement/data/MM-S14-Disease/st21pv_syn_full-all-complexEdge-filt"
!mv "LM-ontology-concept-placement/data/MM-S14-Disease/mention-level-(concept-placement)" "LM-ontology-concept-placement/data/MM-S14-Disease/st21pv_syn_attr-all-complexEdge-unsup"
!mv "LM-ontology-concept-placement/data/MM-S14-Disease/mention-edge-pair-level" "LM-ontology-concept-placement/data/MM-S14-Disease/st21pv_syn_attr-all-complexEdge-edges-final"

!mv "LM-ontology-concept-placement/data/MM-S14-CPP/mention-level-(out-of-KB-discovery)/syn_attr" "LM-ontology-concept-placement/data/MM-S14-CPP/st21pv_syn_attr-all-complexEdge-filt"
!mv "LM-ontology-concept-placement/data/MM-S14-CPP/mention-level-(out-of-KB-discovery)/syn_full" "LM-ontology-concept-placement/data/MM-S14-CPP/st21pv_syn_full-all-complexEdge-filt"
!mv "LM-ontology-concept-placement/data/MM-S14-CPP/mention-level-(concept-placement)" "LM-ontology-concept-placement/data/MM-S14-CPP/st21pv_syn_attr-all-complexEdge-unsup"
!mv "LM-ontology-concept-placement/data/MM-S14-CPP/mention-edge-pair-level" "LM-ontology-concept-placement/data/MM-S14-CPP/st21pv_syn_attr-all-complexEdge-edges-final"

# create folder and move data files
!mkdir LM-ontology-concept-placement/data/MedMentions-preprocessed+
!mkdir LM-ontology-concept-placement/data/MedMentions-preprocessed+/CPP
!mkdir LM-ontology-concept-placement/data/MedMentions-preprocessed+/Disease

!mv LM-ontology-concept-placement/data/MM-S14-CPP/* "LM-ontology-concept-placement/data/MedMentions-preprocessed+/CPP"
!mv LM-ontology-concept-placement/data/MM-S14-Disease/* "LM-ontology-concept-placement/data/MedMentions-preprocessed+/Disease"

# create folder and move ontology files
!mkdir LM-ontology-concept-placement/ontologies
!mv LM-ontology-concept-placement/data/MedMentions-preprocessed+/CPP/ontology/* LM-ontology-concept-placement/ontologies
!mv LM-ontology-concept-placement/data/MedMentions-preprocessed+/Disease/ontology/* LM-ontology-concept-placement/ontologies

# create a sample validation in-KB data file of random 100 data lines (which is need for debugging mode in inference)
!shuf -n 100 LM-ontology-concept-placement/data/MedMentions-preprocessed+/Disease/st21pv_syn_attr-all-complexEdge-edges-final/valid.jsonl > LM-ontology-concept-placement/data/MedMentions-preprocessed+/Disease/st21pv_syn_attr-all-complexEdge-edges-final/valid-sample.jsonl

!mkdir LM-ontology-concept-placement/results

# Validation & Quality Metrics

In [ ]:
class EnrichmentValidator:
    """
    Validate enrichment results and compute quality metrics
    Reference: Dong et al. (2024) - InRany/InRall metrics
    """

    def __init__(self):
        self.validation_results = {}

    def validate_concept_placement(self, new_concepts: List[GeneratedConcept],
                                   existing_concepts: Set[str]) -> Dict:
        """
        Validate concept placement quality
        """
        print("\n🔍 Validating concept placement...")

        metrics = {
            "total_new_concepts": len(new_concepts),
            "valid_parents": 0,
            "valid_definitions": 0,
            "valid_synonyms": 0,
            "high_confidence": 0,
            "issues": []
        }

        for concept in new_concepts:
            # Check parent validity
            for parent in concept.suggested_parents:
                if parent in existing_concepts:
                    metrics["valid_parents"] += 1
                else:
                    metrics["issues"].append({
                        "concept": concept.name,
                        "issue": f"Invalid parent: {parent}"
                    })

            # Check definition
            if len(concept.definition) > 20:
                metrics["valid_definitions"] += 1
            else:
                metrics["issues"].append({
                    "concept": concept.name,
                    "issue": "Definition too short"
                })

            # Check synonyms
            if concept.synonyms:
                metrics["valid_synonyms"] += 1

            # Check confidence
            if concept.confidence > 0.8:
                metrics["high_confidence"] += 1

        # Calculate quality score
        total_checks = len(new_concepts) * 4
        passed_checks = (metrics["valid_parents"] + metrics["valid_definitions"] +
                        metrics["valid_synonyms"] + metrics["high_confidence"])
        metrics["quality_score"] = passed_checks / max(total_checks, 1)

        print(f"   Quality Score: {metrics['quality_score']:.2f}")
        print(f"   Issues found: {len(metrics['issues'])}")

        return metrics

    def compute_placement_metrics(self, gold_edges: List[Tuple],
                                  predicted_edges: List[Tuple]) -> Dict:
        """
        Compute InRany and InRall metrics
        """
        gold_set = set(gold_edges)
        pred_set = set(predicted_edges)

        # InRany: any gold edge found
        inRany = 1 if pred_set & gold_set else 0

        # InRall: all gold edges found
        inRall = 1 if gold_set.issubset(pred_set) else 0

        # Precision, Recall, F1
        intersection = pred_set & gold_set
        precision = len(intersection) / max(len(pred_set), 1)
        recall = len(intersection) / max(len(gold_set), 1)
        f1 = 2 * precision * recall / max(precision + recall, 1)

        return {
            "InRany": inRany,
            "InRall": inRall,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    def generate_validation_report(self, all_results: Dict) -> str:
        """
        Generate comprehensive validation report
        """
        report = f"""
# Ontology Enrichment Validation Report

## Summary
- **New Concepts Generated:** {len(all_results.get('new_concepts', []))}
- **Relations Created:** {len(all_results.get('relations', []))}
- **Resource Links:** {len(all_results.get('resource_links', []))}
- **Cross-Mappings:** {len(all_results.get('cross_mappings', {}))}

## Quality Metrics
| Metric | Value |
|--------|-------|
| Placement Quality Score | {all_results.get('quality_score', 0):.2f} |
| Average Confidence | {all_results.get('avg_confidence', 0):.2f} |
| Valid Parents | {all_results.get('valid_parents', 0)} |
| Valid Definitions | {all_results.get('valid_definitions', 0)} |

## New Concepts
| Concept | Type | Confidence | Parents |
|---------|------|------------|---------|
"""
        for concept in all_results.get('new_concepts', [])[:20]:
            parents = ', '.join(concept.suggested_parents[:2])
            report += f"| {concept.name} | {concept.concept_type} | {concept.confidence} | {parents} |\n"

        return report

#Complete Enrichment Pipeline

In [ ]:
class ProfessionalEnrichmentPipeline:
    """
    Complete professional pipeline for ontology enrichment
    Integrates concept generation, relation creation, and resource linking
    """

    def __init__(self, output_dir: str = "./hlo_enriched"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Initialize components
        self.concept_generator = ProfessionalConceptGenerator(llm_model="gpt-4")
        self.relation_generator = RelationGenerator()
        self.resource_linker = None  # Will be initialized with resources
        self.validator = EnrichmentValidator()

        # Track results
        self.results = {
            "new_concepts": [],
            "relations": [],
            "resource_links": [],
            "cross_mappings": {},
            "validation": {}
        }

    def initialize_resources(self, resources_db: Dict):
        """Initialize resource linker with resources database"""
        self.resource_linker = LearningResourceLinker(resources_db)

    def run_enrichment(self, source_texts: List[Dict],
                      existing_ontology: Dict,
                      resources_db: Dict) -> EnrichmentResult:
        """
        Run complete enrichment pipeline
        """
        print("\n" + "="*70)
        print(" PROFESSIONAL ONTOLOGY ENRICHMENT PIPELINE")
        print("Concept Generation → Relation Creation → Resource Linking")
        print("="*70)

        # ========== STEP 1: Load existing concepts ==========
        print("\n STEP 1: Loading Existing Ontology")
        print("-" * 40)

        existing_concepts = set(existing_ontology.get("concepts", []))
        ontology_graph = existing_ontology.get("graph", {})
        print(f"   ✓ Loaded {len(existing_concepts)} existing concepts")

        # ========== STEP 2: Generate new concepts ==========
        print("\n STEP 2: Generating New Concepts from Texts")
        print("-" * 40)

        new_concepts = self.concept_generator.batch_extract(
            source_texts, existing_concepts
        )
        self.results["new_concepts"] = new_concepts
        print(f"\n    Generated {len(new_concepts)} new concepts")

        # ========== STEP 3: Generate relations ==========
        print("\n STEP 3: Generating Concept Relations")
        print("-" * 40)

        all_relations = []
        existing_list = list(existing_concepts)

        for concept in new_concepts:
            # Generate parents
            parents = self.relation_generator.generate_parent_relations(
                concept, existing_list
            )
            concept.suggested_parents = [p[0] for p in parents[:3]]

            # Generate children
            children = self.relation_generator.generate_child_relations(
                concept, existing_list
            )
            concept.suggested_children = [c[0] for c in children[:3]]

            # Record relations
            for parent, score in parents[:3]:
                all_relations.append((parent, concept.name, "subclass_of", score))
            for child, score in children[:3]:
                all_relations.append((concept.name, child, "subclass_of", score))

            print(f"   {concept.name}: {len(parents[:3])} parents, {len(children[:3])} children")

        self.results["relations"] = all_relations
        print(f"\n    Generated {len(all_relations)} relations")

        # ========== STEP 4: Link to learning resources ==========
        print("\n📖 STEP 4: Linking to Learning Resources")
        print("-" * 40)

        if self.resource_linker:
            resource_links = self.resource_linker.batch_link(new_concepts)
            self.results["resource_links"] = resource_links
            print(f"\n    Created {len(resource_links)} resource links")

            # Display sample links
            for link in resource_links[:5]:
                print(f"      • {link.concept_name} → {link.resource_title} ({link.resource_type}, {link.difficulty_level})")

        # ========== STEP 5: Validate results ==========
        print("\n STEP 5: Validation")
        print("-" * 40)

        validation = self.validator.validate_concept_placement(
            new_concepts, existing_concepts
        )
        self.results["validation"] = validation

        # Compute average confidence
        if new_concepts:
            avg_conf = sum(c.confidence for c in new_concepts) / len(new_concepts)
            self.results["avg_confidence"] = avg_conf
            print(f"   Average Confidence: {avg_conf:.2f}")

        # ========== STEP 6: Generate output ==========
        print("\n STEP 6: Generating Output")
        print("-" * 40)

        # Save results
        self._save_results()

        # Generate report
        report = self.validator.generate_validation_report(self.results)
        report_path = self.output_dir / "ENRICHMENT_REPORT.md"
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report)
        print(f"    Report saved: {report_path}")

        # ========== SUMMARY ==========
        print("\n" + "="*70)
        print(" ENRICHMENT COMPLETED SUCCESSFULLY")
        print("="*70)
        print(f"\n FINAL STATISTICS:")
        print(f"   • New Concepts: {len(new_concepts)}")
        print(f"   • New Relations: {len(all_relations)}")
        print(f"   • Resource Links: {len(resource_links) if resource_links else 0}")
        print(f"   • Quality Score: {validation['quality_score']:.2f}")
        print(f"\n Output Directory: {self.output_dir.absolute()}")
        print("="*70)

        return EnrichmentResult(
            new_concepts=new_concepts,
            concept_relations=all_relations,
            resource_links=resource_links if resource_links else [],
            cross_mappings=self.results["cross_mappings"],
            validation_metrics=validation
        )

    def _save_results(self):
        """Save enrichment results to JSON"""
        import json

        results_path = self.output_dir / "enrichment_results.json"

        # Convert to serializable format
        serializable = {
            "new_concepts": [
                {
                    "name": c.name,
                    "type": c.concept_type,
                    "definition": c.definition,
                    "synonyms": c.synonyms,
                    "confidence": c.confidence,
                    "suggested_parents": c.suggested_parents,
                    "suggested_children": c.suggested_children
                }
                for c in self.results["new_concepts"]
            ],
            "relations": self.results["relations"],
            "resource_links": [
                {
                    "concept": l.concept_name,
                    "resource": l.resource_title,
                    "type": l.resource_type,
                    "difficulty": l.difficulty_level,
                    "score": l.match_score
                }
                for l in self.results["resource_links"]
            ],
            "validation": self.results["validation"]
        }

        with open(results_path, 'w', encoding='utf-8') as f:
            json.dump(serializable, f, indent=2, ensure_ascii=False)

        print(f"   ✓ Results saved: {results_path}")


# ============================================
# MAIN EXECUTION
# ============================================

def main():
    """
    Example execution of professional enrichment pipeline
    """
    # Sample medical texts for concept extraction
    source_texts = [
        {
            "content": """
            Long COVID, also known as post-acute sequelae of SARS-CoV-2 infection (PASC),
            is characterized by persistent symptoms lasting more than 12 weeks. Common symptoms
            include severe fatigue, cognitive impairment (brain fog), dyspnea, chest pain,
            and palpitations. Management requires multidisciplinary approach including
            symptom-based treatment and rehabilitation.
            """,
            "context": "New emerging disease concept"
        },
        {
            "content": """
            Digital therapeutics (DTx) are evidence-based therapeutic interventions delivered
            through software applications. They have shown efficacy in diabetes management,
            hypertension control, and mental health disorders including depression and anxiety.
            """,
            "context": "Digital health intervention"
        }
    ]

    # Mock existing ontology concepts
    existing_ontology = {
        "concepts": [
            "COVID-19", "SARS-CoV-2 infection", "Fatigue", "Dyspnea", "Chest pain",
            "Diabetes mellitus", "Hypertension", "Depression", "Anxiety disorder",
            "Rehabilitation", "Multidisciplinary care"
        ],
        "graph": {}
    }

    # Mock resources database
    resources_db = {
        "videos": [],
        "articles": [],
        "guidelines": []
    }

    # Run enrichment
    pipeline = ProfessionalEnrichmentPipeline(output_dir="./hlo_enriched_output")
    pipeline.initialize_resources(resources_db)

    results = pipeline.run_enrichment(
        source_texts=source_texts,
        existing_ontology=existing_ontology,
        resources_db=resources_db
    )

    # Print sample results
    print("\n SAMPLE GENERATED CONCEPTS:")
    for concept in results.new_concepts[:3]:
        print(f"\n   • {concept.name} ({concept.concept_type})")
        print(f"     Definition: {concept.definition[:100]}...")
        print(f"     Parents: {', '.join(concept.suggested_parents[:2])}")
        print(f"     Confidence: {concept.confidence}")

    print("\n SAMPLE RESOURCE LINKS:")
    for link in results.resource_links[:5]:
        print(f"\n   • {link.concept_name}")
        print(f"     → {link.resource_title}")
        print(f"     Type: {link.resource_type}, Level: {link.difficulty_level}")

if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 PROFESSIONAL ONTOLOGY ENRICHMENT PIPELINE
Concept Generation → Relation Creation → Resource Linking

 STEP 1: Loading Existing Ontology
----------------------------------------
   ✓ Loaded 11 existing concepts

 STEP 2: Generating New Concepts from Texts
----------------------------------------

📄 Processing text 1/2

🔍 Extracting concepts from text...
   ⚠️ LLM API error: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


📄 Processing text 2/2

🔍 Extracting concepts from text...
   ⚠️ LLM API error: 

You tried to access openai.ChatCompletion, but this is no longer supporte

# Builer HLO

In [ ]:
!git clone https://github.com/asgmodel/HLO.git

Cloning into 'HLO'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 14 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 23.75 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
!python /content/HLO/ontology_config.py --template research


🏥 HEALTH LEARNING ONTOLOGY - CONCEPT CONFIGURATION

Select configuration mode:
  1. Interactive configuration (manual entry)
  2. Load configuration from file
  3. Use predefined template
  4. Display all available sources

Enter your choice (1-4): 3

Available templates:
  1. Clinical Focus (diagnosis, treatment, procedures)
  2. Research Focus (literature indexing, discovery)
  3. Digital Health Focus (informatics, technology)
  4. Comprehensive (balanced coverage)
  5. Light (for testing)
  6. Ultra Large (maximum concepts)

Select template (1-6): 1
✅ SNOMED_CT: 500 concepts allocated
✅ ICD_11: 200 concepts allocated
✅ ICD_10: 150 concepts allocated
✅ RxNorm: 150 concepts allocated
✅ CPT: 100 concepts allocated
✅ LOINC: 100 concepts allocated
✅ MeSH: 100 concepts allocated
✅ ATC: 50 concepts allocated
✅ MIMO: 20 concepts allocated
✅ HPO: 20 concepts allocated
✅ MONDO: 20 concepts allocated
✅ Custom: 50 concepts allocated

📊 Total concepts: 1460

ONTOLOGY CONCEPT ALLOCATION REPORT



In [ ]:
!python /content/HLO/builder_hlo.py


HEALTH LEARNING ONTOLOGY (HLO) - PROFESSIONAL OWL GENERATOR
Start Time: 20:40:12
Status: Generating OWL 2 DL compliant ontology

Generating OWL ontology content...

Writing OWL Header: |██████████████████████████████████████████████████| 100.0% [1/1] ETA: 0.0s
Writing OWL Header: |████████████████████████████████████████████████████████████████████████████████████████████████████| 200.0% [2/1] ETA: -0.0s
Writing Top Level Classes: |██████████████████████████████████████████████████| 100.0% [8/8] ETA: 0.0s
Writing Top Level Classes: |████████████████████████████████████████████████████████████████████████████████████████████████████| 200.0% [16/8] ETA: -0.0s
Writing Disease Classes: |██████████████████████████████████████████████████| 100.0% [48/48] ETA: 0.0s
Writing Disease Classes: |████████████████████████████████████████████████████████████████████████████████████████████████████| 200.0% [96/48] ETA: -0.0s
Writing Symptom Classes: |██████████████████████████████████████████████████

In [ ]:
!python /content/HLO/generator_hlo.py


HEALTH LEARNING ONTOLOGY (HLO) - INSTANT GENERATION SYSTEM
Start Time: 20:40:12

Generating concepts: |████████████████████████████████████████| 100.0% [1105/1105] ETA: 0.0s
Generating concepts: |████████████████████████████████████████| 100.0% [1105/1105] ETA: 0.0s
Generating concepts: |████████████████████████████████████████| 100.0% [1105/1105] ETA: 0.0s
Generating relations: |████████████████████████████████████████| 100.0% [1250/1250] ETA: 0.0s
Generating relations: |████████████████████████████████████████| 100.0% [1250/1250] ETA: 0.0s
Generating relations: |████████████████████████████████████████| 100.0% [1250/1250] ETA: 0.0s
Generating resources: |████████████████████████████████████████| 100.0% [1200/1200] ETA: 0.0s
Generating resources: |████████████████████████████████████████| 100.0% [1200/1200] ETA: 0.0s
Generating resources: |████████████████████████████████████████| 100.0% [1200/1200] ETA: 0.0s


Writing OWL file: |████████████████████████████████████████| 100.0% [3/3]